In [ ]:
import numpy as np
import ssqpy
from time import time, sleep

np.set_printoptions(suppress=True)

# Build MPC
ssqpy.setSilentMode()

dt = 0.01
MPC_H = 10

V_WGT = np.array((6e-1, 2.4e-1))
U_WGT = np.array((0.0, 3e-2))
ELBOW_WGT = 200.0
TIP_WGT = 120.0

elbow_height = 0.1
tip_height = 0.2

torque_clip = 0.08

vel_bounds = np.array((30, 25))
torque_bounds = np.array((0.005, torque_clip,))

model = ssqpy.model.Model(
    MPC_H,
    dt,
    urdf_path="acrobot.urdf",
    solver_mode=ssqpy.model.SolverMode.InverseDynamics,
)

nq = model.getnq()
nv = model.getnv()
nu = model.getnu()

vel_cost = ssqpy.model.costs.SquaredJointVelocityCost(model, V_WGT)
u_cost = ssqpy.model.costs.SquaredControlCost(model, U_WGT)
elbow_cost = ssqpy.model.costs.FrameSquaredTranslationErrorCost(
    model, "link2", np.array((0.0, 0.0, elbow_height)), ELBOW_WGT
)
tip_cost = ssqpy.model.costs.FrameSquaredTranslationErrorCost(
    model, "tip", np.array((0.0, 0.0, tip_height)), TIP_WGT
)

for k in range(MPC_H):
    model.addCost(k, vel_cost)
    model.addCost(k, u_cost)
    model.addCost(k, elbow_cost)
    model.addCost(k, tip_cost)

model.addCost(MPC_H, vel_cost)
model.addCost(MPC_H, elbow_cost)
model.addCost(MPC_H, tip_cost)

model.finalize(
    custom_velocity_bounds=vel_bounds,
    custom_torque_bounds=torque_bounds,
)

ssqp_params = ssqpy.solvers.ssqpParams()
ssqp_params.tolerance = 1e-2
ssqp_params.qp_solver_type = ssqpy.solvers.QPSolverType.HPIPM

hpipm_params = ssqpy.solvers.hpipmParams()
hpipm_params.tol_comp = 1e-3
hpipm_params.tol_stat = 1e-3
hpipm_params.tol_eq = 1e-3
hpipm_params.tol_ineq = 1e-3
ssqp_params.hpipmParams = hpipm_params

mpc = ssqpy.solvers.MPC(model, ssqp_params, sqp_iters=6, qp_iters=100)

In [ ]:
from cloudpendulumclient.client import Client

user_token = "MY_TOKEN"

Tf = 60.0
client = Client()
session_token, livestream_url = client.start_experiment(
    user_token = user_token,
    experiment_type = "DoublePendulum",
    experiment_time = Tf,
    preparation_time = 5.0,
    record = True
)

print("Received response from server!")
print("Session token: ", session_token)
print("Livestream url: ", livestream_url)

current_time = 0.0

np.set_printoptions(suppress=True)

start_all = time()
while (time() - start_all) < Tf:    
    mq = client.get_position(session_token)
    mv = client.get_velocity(session_token)
    
    try:
        u = mpc.step(np.hstack((mq, mv)))[1].stage(0)
    except RuntimeError:
        u = np.zeros(nv)

    tau = model.inverseDynamics(np.array(mq), np.array(mv), u)
    tau = np.clip(tau, -torque_bounds, torque_bounds)

    try:
        client.set_torque(tau, session_token)
    except RuntimeError as err:
        print(err)
        break

url = client.stop_experiment(session_token)

print("Final state:", mq)